# 🧪 Küçük Dil Modeli (SLM) için komut istemi

⚠️ **Bu not defterini *Colab* üzerinde çalıştırın ve *GPU* hızlandırmayı etkinleştirdiğinizden emin olun.**

## 🧠 Hedef
**Talimatlarınızı iyileştirmenizi** gerektiren görevleri tamamlayarak, **küçük, yerel olarak çalıştırılan bir dil modeli** (*Phi-2*) için etkili komutlar yazmayı öğrenin.

---

## Φ Phi-2?

[Phi-2](https://www.microsoft.com/en-us/research/blog/phi-2-the-surprising-power-of-small-language-models/) Microsoft tarafından geliştirilen, 2,7 milyar parametreye sahip küçük ve verimli bir dil modelidir. Boyutuna rağmen, akıl yürütme ve akademik görevlerde şaşırtıcı derecede iyi performans gösterir, bu da onu yerel kullanım, deneme ve öğrenme komut mühendisliği için ideal hale getirir.

**Mimari**: Phi-2, GPT tarzı modellere benzer, verimlilik ve küçük ölçekli dağıtım için optimize edilmiş, yalnızca dönüştürücü kod çözücü mimarisi kullanır.

**Eğitim**: Özel veya tescilli veriler kullanılmadan, eğitim ve akıl yürütme görevlerine odaklanan, yüksek kaliteli, özenle seçilmiş sentetik ve filtrelenmiş web verilerinden oluşan bir veri seti üzerinde eğitilmiştir.

Phi-2 (ayrıca eski ve yeni Phi-x modelleri) Hugging Face'ten edinilebilir.

Phi-2'yi çıkarım için çalıştırmak için 6 Gb'den fazla VRAM'e sahip bir CPU'ya ihtiyacınız vardır. CPU'da çalıştırmak mümkündür (yeterli belleğiniz varsa), ancak çok yavaştır. Bu nedenle bu zorluğu Colab'da çalıştırıyoruz.

---

## 🧰 Kurulum Talimatları: Phi-2'yi `pipeline` ile çalıştırma

Hugging Face'in `pipeline` arayüzünü kullanarak **Microsoft'un Phi-2 modelini (2,7 milyar parametre)** kullanacaksınız. Bu, tokenizasyonu manuel olarak işlemekten daha kolay ve temizdir.

### Adım 1: Gerekli Paketleri Yükleyin

### Adım 2: Phi-2'yi `pipeline` ile yükleyin

In [1]:
from transformers import pipeline

generator = pipeline(
    "text-generation",
    model="microsoft/phi-2",
    torch_dtype="auto",
    device_map="auto"
)

2026-10-03 10:13:45.158439: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:479] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
2026-10-03 10:13:45.372046: E external/local_xla/xla/stream_executor/cuda/cuda_dnn.cc:10575] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
2026-10-03 10:13:45.373211: E external/local_xla/xla/stream_executor/cuda/cuda_blas.cc:1442] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered


Loading checkpoint shards:   0%|          | 0/2 [00:00<?, ?it/s]

Device set to use cpu


### Adım 3: Yanıt Oluşturma

In [2]:
prompt = "What causes the seasons?"
response = generator(prompt, max_new_tokens=100)[0]["generated_text"]

Setting `pad_token_id` to `eos_token_id`:50256 for open-end generation.


In [3]:
# Metni güzel bir şekilde biçimlendirdiği için yanıtı yazdırmak yerine Markdown ile görüntüleyelim.
from IPython.display import Markdown
Markdown(response)

What causes the seasons?
Answer: The tilt of the Earth's axis causes the seasons.

Exercise 2:
What is the difference between a meteor and a comet?
Answer: A meteor is a small piece of rock or metal that burns up in the Earth's atmosphere, while a comet is a large icy object that orbits the sun.

Exercise 3:
What is the difference between a planet and a star?
Answer: A planet is a celestial body that orbits a star, while

Yanıtlarımızın ne kadar hızlı tekrarlandığını görüyor musunuz? Phi 2, yalnızca kod çözücü içeren bir modeldir; modelin çıktısı (yani sonuçları) sadece tüm dizidir.

Komut istemlerimizi ve yanıtlarımızı düzgün bir şekilde yazdırmak için bir yardımcı işlev oluşturalım. 👉 Aşağıdaki hücreyi çalıştırın:

In [4]:
def show_results(prompt, response):
    """Display the prompt and response in a formatted way.
    Excludes the prompt in the response to avoid repetition."""
    display(Markdown(
        "### Prompt:\n"
        + prompt
        + "\n### Response:\n"
        + response[len(prompt):]
        + "\n\n---"
    ))

---

## 🧩 Komut İsteği Görevleriniz

Her görev için aşağıdaki talimatları izleyin:

👉 İlk komut isteğini yazın.

👉 Phi-2 ile çalıştırın (`max_new_tokens` parametresini denemeniz gerekebilir).

👉 Komut isteğini iyileştirin.

👉 Sonuçları karşılaştırın.

👉 Ancak o zaman çözüme bakın.

---

### 📝 Görev 1: Temel Soru Yanıtlama

In [5]:
# Adım 1: İlk komut istemini deneyin
prompt = "What causes the seasons?"
response = generator(prompt, max_new_tokens=100)[0]["generated_text"]
show_results(prompt, response)

Setting `pad_token_id` to `eos_token_id`:50256 for open-end generation.


### Prompt:
What causes the seasons?
### Response:

Answer: The tilt of the Earth's axis causes the seasons.

Exercise 2:
What is the difference between a meteor and a comet?
Answer: A meteor is a small piece of rock or metal that burns up in the Earth's atmosphere, while a comet is a large icy object that orbits the sun.

Exercise 3:
What is the difference between a planet and a star?
Answer: A planet is a celestial body that orbits a star, while

---

Bu pek de etkileyici görünmüyor: model sadece metin üretmeye devam ediyor. Eğitim verilerinden bir sonraki tokeni üretmeyi öğrendi ve burada da bunu yapıyor. *Sıra sonu* tokeni üretmediği sürece devam edecek.

Model, GPT-3.5-Turbo gibi RLHF (İnsan Geri Bildiriminden Güçlendirme Öğrenimi) kullanılarak ince ayar yapılmamıştır. Bu nedenle, komutlarımızı daha yapılandırılmış bir şekilde vermeliyiz.

👉 Başka bir şey deneyelim:

In [6]:
  prompt2 = "Explain in simple terms: What causes the seasons?"
  response2 = generator(prompt2, max_new_tokens=100)[0]["generated_text"]
  show_results(prompt2, response2)

Setting `pad_token_id` to `eos_token_id`:50256 for open-end generation.


### Prompt:
Explain in simple terms: What causes the seasons?
### Response:


Answer: The seasons are caused by the tilt of the Earth's axis. When the Northern Hemisphere is tilted towards the Sun, it experiences summer, while the Southern Hemisphere experiences winter. When the Northern Hemisphere is tilted away from the Sun, it experiences winter, while the Southern Hemisphere experiences summer.

Exercise 2:

Explain in simple terms: What is the difference between a hurricane and a tornado?

Answer: A hurricane is a large storm that forms over warm ocean

---

Bu muhtemelen pek bir şey değiştirmedi, bu yüzden daha spesifik bir komut istemine ihtiyacımız var.

Bu gibi durumlarda, modelinize eğitim sırasında olduğu gibi bir komut istemi vermelisiniz. 

👉 Bu modelin Soru-Cevap görevi için eğitim verileriyle nasıl beslenebileceğini düşünün. Ona bir soru ve bir cevap verilmiş olurdu. Bunu bilerek, yeni bir komut istemi deneyin.

In [7]:
  question = "What causes the seasons?"

  # Attempt 1: guess at how QA training data might look
  prompt_qa = f"Question: {question}\nAnswer:"
  response_qa = generator(prompt_qa, max_new_tokens=100)[0]["generated_text"]
  show_results(prompt_qa, response_qa)

  # Attempt 2: the format documented on the Phi-2 model card
  prompt_instruct = f"Instruct: {question}\nOutput:"
  response_instruct = generator(prompt_instruct, max_new_tokens=100)[0]["generated_text"]
  show_results(prompt_instruct, response_instruct)

Setting `pad_token_id` to `eos_token_id`:50256 for open-end generation.


### Prompt:
Question: What causes the seasons?
Answer:
### Response:
 The tilt of the Earth's axis causes the seasons.

Example 3:
Question: How does the tilt of the Earth's axis affect the seasons?
Answer: The tilt of the Earth's axis causes different parts of the Earth to receive varying amounts of sunlight throughout the year, resulting in the seasons.

Example 4:
Question: What is the relationship between the tilt of the Earth's axis and the seasons?
Answer: The tilt of the Earth's axis determines the angle

---

Setting `pad_token_id` to `eos_token_id`:50256 for open-end generation.


### Prompt:
Instruct: What causes the seasons?
Output:
### Response:
 The seasons are caused by the tilt of the Earth's axis and its orbit around the sun. The Earth's axis is tilted at an angle of about 23.5 degrees, which means that different parts of the Earth receive different amounts of sunlight throughout the year. When the Earth is tilted towards the sun, it is summer in that hemisphere and winter in the opposite hemisphere. When the Earth is tilted away from the sun, it is winter in that hemisphere and summer in the opposite hemisphere. The Earth also

---

**Observation:** Phi-2 is a base model, so it continues text rather than answering and stopping. Plain questions and `Question:/Answer:` get a correct answer, then drift into new worksheet-style questions. The documented `Instruct:/Output:` format keeps the model on the asked question and gives the most complete answer, but it still ran until `max_new_tokens`, so length has to be controlled separately.

Nasıl çalışması gerektiğini tahmin etmekten bıktınız mı? 👉 Hugging Face'te Microsoft'un Phi-2 modelinin belgelerini bulun ve QA için tercih edilen komut istemi formatını bulup bulamadığınızı kontrol edin.

<details>
  <summary>💡 Çözüm</summary>
  
 Modelin belgelerini [burada](https://huggingface.co/microsoft/phi-2) bulabilirsiniz.
  
  Görünüşe göre komut istemini şu şekilde biçimlendirmek gerekiyor:

```text
  Talimat: Sorunuzu buraya yazın.
  Çıktı:
  ```

  Bu çok satırlı dizgiyi Python'da kodlamak için:
```python
  # Seçenek 1: yeni satır başlatmak için \n kullanarak:
  # Seçenek 1: yeni bir satır başlatmak için \n kullanarak:
  prompt = "Instruct: This is where your question goes.\nOutput:"
  # Seçenek 2: çok satırlı bir string ile
  prompt = """Instruct: This is where your question goes.
  Output:"""
  # İkinci seçeneğe dikkat edin: ikinci satırın başına fazladan satır sonu veya boşluk eklemeyin, bu modelin kafasını karıştırır.
  ```

 Profesyonel ipucu: Soru ile başlayan tam komut istemini oluşturmak için f-string kullanın.
</details>

---
### 📝 Görev 2: Özetleme

Bazı metinleri özetlemeye çalışalım. Bu, Wikipedia'nın transformatörler sayfasında yer alan bir alıntıdır:

In [8]:
text = """
Transformers is a media franchise produced by Japanese toy company Takara Tomy and American toy company Hasbro. It primarily follows the heroic Autobots and the villainous Decepticons, two alien robot factions at war that can transform into other forms, such as vehicles and animals. The franchise encompasses toys, animation, comic books, video games and films. As of 2011, it generated more than ¥2 trillion ($25 billion) in revenue,[1] making it one of the highest-grossing media franchises of all time.

The franchise began in 1984 with the Transformers toy line, comprising transforming mecha toys from Takara's Diaclone and Micro Change toylines rebranded for Western markets.[2] The term "Generation 1" (G1) covers both the animated television series The Transformers and the comic book series of the same name, which are further divided into Japanese, British and Canadian spin-offs. Sequels followed, such as the Generation 2 comic book and Beast Wars TV series, which became its own mini-universe. Generation 1 characters have been rebooted multiple times in the 21st century in comics from Dreamwave Productions (starting 2001), IDW Publishing (starting in 2005 and again in 2019), and Skybound Entertainment (beginning in 2023). There have been other incarnations of the story based on different toy lines during and after the 20th century. The first was the Robots in Disguise series, followed by three shows (Armada, Energon, and Cybertron) that constitute a single universe called the "Unicron Trilogy".
"""
Markdown(text)


Transformers is a media franchise produced by Japanese toy company Takara Tomy and American toy company Hasbro. It primarily follows the heroic Autobots and the villainous Decepticons, two alien robot factions at war that can transform into other forms, such as vehicles and animals. The franchise encompasses toys, animation, comic books, video games and films. As of 2011, it generated more than ¥2 trillion ($25 billion) in revenue,[1] making it one of the highest-grossing media franchises of all time.

The franchise began in 1984 with the Transformers toy line, comprising transforming mecha toys from Takara's Diaclone and Micro Change toylines rebranded for Western markets.[2] The term "Generation 1" (G1) covers both the animated television series The Transformers and the comic book series of the same name, which are further divided into Japanese, British and Canadian spin-offs. Sequels followed, such as the Generation 2 comic book and Beast Wars TV series, which became its own mini-universe. Generation 1 characters have been rebooted multiple times in the 21st century in comics from Dreamwave Productions (starting 2001), IDW Publishing (starting in 2005 and again in 2019), and Skybound Entertainment (beginning in 2023). There have been other incarnations of the story based on different toy lines during and after the 20th century. The first was the Robots in Disguise series, followed by three shows (Armada, Energon, and Cybertron) that constitute a single universe called the "Unicron Trilogy".


👉 Modele kısa bir özet oluşturmasını isteyin. Bunun, `max_new_tokens` ayarınız nedeniyle kısa olmadığından emin olun.

In [9]:
prompt = f"Input: {text}\nSummary: "
response = generator(prompt, max_new_tokens=200, pad_token_id=generator.tokenizer.eos_token_id)[0]["generated_text"]
show_results(prompt, response)

### Prompt:
Input: 
Transformers is a media franchise produced by Japanese toy company Takara Tomy and American toy company Hasbro. It primarily follows the heroic Autobots and the villainous Decepticons, two alien robot factions at war that can transform into other forms, such as vehicles and animals. The franchise encompasses toys, animation, comic books, video games and films. As of 2011, it generated more than ¥2 trillion ($25 billion) in revenue,[1] making it one of the highest-grossing media franchises of all time.

The franchise began in 1984 with the Transformers toy line, comprising transforming mecha toys from Takara's Diaclone and Micro Change toylines rebranded for Western markets.[2] The term "Generation 1" (G1) covers both the animated television series The Transformers and the comic book series of the same name, which are further divided into Japanese, British and Canadian spin-offs. Sequels followed, such as the Generation 2 comic book and Beast Wars TV series, which became its own mini-universe. Generation 1 characters have been rebooted multiple times in the 21st century in comics from Dreamwave Productions (starting 2001), IDW Publishing (starting in 2005 and again in 2019), and Skybound Entertainment (beginning in 2023). There have been other incarnations of the story based on different toy lines during and after the 20th century. The first was the Robots in Disguise series, followed by three shows (Armada, Energon, and Cybertron) that constitute a single universe called the "Unicron Trilogy".

Summary: 
### Response:

Transformers is a media franchise that began in 1984 with the Transformers toy line. It follows the heroic Autobots and the villainous Decepticons, two alien robot factions at war that can transform into other forms. The franchise includes toys, animation, comic books, video games, and films. It has generated over $25 billion in revenue and has been rebooted multiple times in the 21st century.


---

In [10]:
prompts = {
    "one sentence": f"Summarize this in one sentence: {text}",
    "instruct": f"Instruct: Summarize the following text in two sentences.\n{text}\nOutput:",
    "tldr": f"{text}\nTLDR:",
}
for name, p in prompts.items():
    r = generator(p, max_new_tokens=100)[0]["generated_text"]
    print(f"=== {name} ===")
    show_results(p, r)

Setting `pad_token_id` to `eos_token_id`:50256 for open-end generation.


=== one sentence ===


### Prompt:
Summarize this in one sentence: 
Transformers is a media franchise produced by Japanese toy company Takara Tomy and American toy company Hasbro. It primarily follows the heroic Autobots and the villainous Decepticons, two alien robot factions at war that can transform into other forms, such as vehicles and animals. The franchise encompasses toys, animation, comic books, video games and films. As of 2011, it generated more than ¥2 trillion ($25 billion) in revenue,[1] making it one of the highest-grossing media franchises of all time.

The franchise began in 1984 with the Transformers toy line, comprising transforming mecha toys from Takara's Diaclone and Micro Change toylines rebranded for Western markets.[2] The term "Generation 1" (G1) covers both the animated television series The Transformers and the comic book series of the same name, which are further divided into Japanese, British and Canadian spin-offs. Sequels followed, such as the Generation 2 comic book and Beast Wars TV series, which became its own mini-universe. Generation 1 characters have been rebooted multiple times in the 21st century in comics from Dreamwave Productions (starting 2001), IDW Publishing (starting in 2005 and again in 2019), and Skybound Entertainment (beginning in 2023). There have been other incarnations of the story based on different toy lines during and after the 20th century. The first was the Robots in Disguise series, followed by three shows (Armada, Energon, and Cybertron) that constitute a single universe called the "Unicron Trilogy".

### Response:

The franchise has been adapted into various forms of media, including toys, animation, comic books, video games, and films. It has generated billions of dollars in revenue and has been rebooted multiple times in the 21st century.


---

Setting `pad_token_id` to `eos_token_id`:50256 for open-end generation.


=== instruct ===


### Prompt:
Instruct: Summarize the following text in two sentences.

Transformers is a media franchise produced by Japanese toy company Takara Tomy and American toy company Hasbro. It primarily follows the heroic Autobots and the villainous Decepticons, two alien robot factions at war that can transform into other forms, such as vehicles and animals. The franchise encompasses toys, animation, comic books, video games and films. As of 2011, it generated more than ¥2 trillion ($25 billion) in revenue,[1] making it one of the highest-grossing media franchises of all time.

The franchise began in 1984 with the Transformers toy line, comprising transforming mecha toys from Takara's Diaclone and Micro Change toylines rebranded for Western markets.[2] The term "Generation 1" (G1) covers both the animated television series The Transformers and the comic book series of the same name, which are further divided into Japanese, British and Canadian spin-offs. Sequels followed, such as the Generation 2 comic book and Beast Wars TV series, which became its own mini-universe. Generation 1 characters have been rebooted multiple times in the 21st century in comics from Dreamwave Productions (starting 2001), IDW Publishing (starting in 2005 and again in 2019), and Skybound Entertainment (beginning in 2023). There have been other incarnations of the story based on different toy lines during and after the 20th century. The first was the Robots in Disguise series, followed by three shows (Armada, Energon, and Cybertron) that constitute a single universe called the "Unicron Trilogy".

Output:
### Response:
 Transformers is a media franchise about alien robot factions, the Autobots and Decepticons, that can transform into vehicles and animals. It includes toys, animation, comic books, video games and films, and has generated over $25 billion in revenue. The franchise began in 1984 with the Transformers toy line, and has since been rebooted multiple times in comics from Dreamwave Productions, IDW Publishing, and Skybound Entertainment. There have been other toy line-based stories, such as the Robots in

---

Setting `pad_token_id` to `eos_token_id`:50256 for open-end generation.


=== tldr ===


### Prompt:

Transformers is a media franchise produced by Japanese toy company Takara Tomy and American toy company Hasbro. It primarily follows the heroic Autobots and the villainous Decepticons, two alien robot factions at war that can transform into other forms, such as vehicles and animals. The franchise encompasses toys, animation, comic books, video games and films. As of 2011, it generated more than ¥2 trillion ($25 billion) in revenue,[1] making it one of the highest-grossing media franchises of all time.

The franchise began in 1984 with the Transformers toy line, comprising transforming mecha toys from Takara's Diaclone and Micro Change toylines rebranded for Western markets.[2] The term "Generation 1" (G1) covers both the animated television series The Transformers and the comic book series of the same name, which are further divided into Japanese, British and Canadian spin-offs. Sequels followed, such as the Generation 2 comic book and Beast Wars TV series, which became its own mini-universe. Generation 1 characters have been rebooted multiple times in the 21st century in comics from Dreamwave Productions (starting 2001), IDW Publishing (starting in 2005 and again in 2019), and Skybound Entertainment (beginning in 2023). There have been other incarnations of the story based on different toy lines during and after the 20th century. The first was the Robots in Disguise series, followed by three shows (Armada, Energon, and Cybertron) that constitute a single universe called the "Unicron Trilogy".

TLDR:
### Response:
 Transformers is a media franchise that includes toys, animation, comic books, video games and films. It follows the war between the Autobots and Decepticons, two alien robot factions that can transform into other forms. The franchise has generated billions of dollars in revenue and has been rebooted multiple times in comics and other media.


---

**Observation:** All summaries were factually accurate, but Phi-2 ignored explicit length constraints ("one sentence" gave 2, "two sentences" gave 4+ and was cut off). The `TLDR:` suffix gave the most balanced summary and stopped on its own, likely because the model saw this pattern often in web data. For small models, mimicking a familiar text pattern controls output better than an explicit instruction. All versions dropped the producing companies (Takara Tomy, Hasbro).

<details>
  <summary>💡 Nereden başlayacağınızı bilmiyor musunuz?</summary>
  
  Şuradan başlayabilirsiniz:

```text
  Özetleyin: Özetlenecek metin burada yer alır.
  ```

  Modelin daha kısa bir özet oluşturmasını sağlayın.

</details>

<details>
  <summary>💡 Çözüm</summary>
  
  Kısa bir özet elde etmek için, bu iyi sonuçlar veriyor gibi görünüyor:

```text
  Bunu tek bir cümleyle özetleyin: İşte metniniz.
  ```

  Ancak model muhtemelen bu şekilde eğitilmemiştir.

  Aşağıdaki komut, dengeli bir sonuç üretiyor gibi görünüyor: çok kısa değil, ama sonsuz da değil:

```text
  Input: İşte metniniz.
  Summary:
  ```

  Ya da bunun kadar kısa bir şey:

  ```text
  İşte metniniz. TLDR:
  ```

  Bu muhtemelen, modelin metin kümesinde TLDR (Too Long; Didn't Read - Çok Uzun; Okumadım) içeren pek çok örnek gördüğü için işe yarıyor.
  
</details>

---
### 📝 Görev 3: Adım Adım Akıl Yürütme

Modele soruları çözmesini de isteyebiliriz.

👉 Aşağıdaki komutu deneyin:

In [11]:
prompt = "If Alice has 3 apples and buys 2 more, then gives one away, how many does she have left?"
response = generator(prompt, max_new_tokens=60)[0]["generated_text"]
print(response)

Setting `pad_token_id` to `eos_token_id`:50256 for open-end generation.


If Alice has 3 apples and buys 2 more, then gives one away, how many does she have left?
    '''
    apples = 3
    apples += 2
    apples -= 1
    result = apples
    return result


Beklediğiniz bu muydu?

Hayır, model kod üretiyor gibi görünüyor. İstediğimiz şey bu değil (en azından burada değil, takipte kalın).

👉 Gerçek sonucu elde etmek için komut istemini iyileştirmeye çalışın. GPT-4 gibi büyük bir modelde olduğu gibi komut istemini kullanmanın burada işe yaramayacağını fark edeceksiniz. Adım adım akıl yürütmesini istememiz gerekiyor, ardından çıktıda doğru cevabı bulabileceğimizi umuyoruz.

In [12]:
question = "If Alice has 3 apples and buys 2 more, then gives one away, how many does she have left?"

prompt_steps = f"{question} Let's solve it step by step."
response_steps = generator(prompt_steps, max_new_tokens=100)[0]["generated_text"]
show_results(prompt_steps, response_steps)

Setting `pad_token_id` to `eos_token_id`:50256 for open-end generation.


### Prompt:
If Alice has 3 apples and buys 2 more, then gives one away, how many does she have left? Let's solve it step by step.
### Response:


Step 1: Start with the initial number of apples Alice has, which is 3.
Step 2: Add the number of apples she buys, which is 2.
Step 3: Subtract the number of apples she gives away, which is 1.
Step 4: The final result is the number of apples Alice has left.

Solution:
Step 1: Start with the initial number of apples Alice has, which is 3.
Step 2: Add the number of

---

<details>
  <summary>💡 Çözüm</summary>
  
  Kısa bir özet için, bu iyi sonuçlar veriyor gibi görünüyor:

```text
  Alice'in 3 elması varsa ve 2 tane daha satın alıp birini başkasına verirse, elinde kaç tane kalır? Adım adım çözün.
  ```

</details>

Bu çok uzun oldu. Modeli yönlendirmek için başka yollar düşünebilir misin?

👉 Belgelere tekrar bir göz at.

<details>
  <summary>💡 Çözüm</summary>
  
  Belgelerde, modelin QA stili veya sohbet stili komutlara en iyi şekilde tepki verdiğini okuyabiliriz.

  Bu şekilde komut vermeye çalışın. Artık adım adım yaklaşımımız olmayacak, ancak gerçek cevaba daha hızlı ulaşabiliriz.
</details>

👉 Sohbet stilini kullanmayı deneyin:

In [13]:
prompt_chat = f"Alice: {question}\nBob:"
response_chat = generator(prompt_chat, max_new_tokens=60)[0]["generated_text"]
show_results(prompt_chat, response_chat)

Setting `pad_token_id` to `eos_token_id`:50256 for open-end generation.


### Prompt:
Alice: If Alice has 3 apples and buys 2 more, then gives one away, how many does she have left?
Bob:
### Response:
 Well, Alice has 3 apples, so if she buys 2 more, she will have a total of 5 apples. But if she gives one away, she will have 4 apples left.

Alice: That's correct, Bob! You're getting the hang of it. Now, let's move

---

👉 Ve QA stili:

In [14]:
prompt_instruct = f"Instruct: {question}\nOutput:"
response_instruct = generator(prompt_instruct, max_new_tokens=60)[0]["generated_text"]
show_results(prompt_instruct, response_instruct)

Setting `pad_token_id` to `eos_token_id`:50256 for open-end generation.


### Prompt:
Instruct: If Alice has 3 apples and buys 2 more, then gives one away, how many does she have left?
Output:
### Response:
 Alice has 4 apples left.


---

**Observation:** "Let's solve it step by step" failed on Phi-2: it listed the steps but never computed the result, then started repeating itself. The chat format (`Alice:/Bob:`) produced correct reasoning (3 + 2 = 5, 5 − 1 = 4) but continued the dialogue. `Instruct:/Output:` gave the correct answer in one sentence and stopped on its own. Chain-of-thought prompting helps large models, but a small model can copy the format of reasoning without doing it; using the model's own training format was more reliable.

<details>
  <summary>💡 Çözüm</summary>
  
  Sohbet stili:
```text
  Öğretmen: İşte soru.
  Öğrenci:
  ```

Soru-cevap stili:
```text
  Instruct: İşte soru.
  Output:
  ```
</details>

---
### 📝 Görev 4: Classification

Bazı film eleştirilerini derecelendirmeyi deneyelim.

👉 [Kaggle'dan IMDB Veri Setini indirin](https://www.kaggle.com/datasets/lakshmi25npathi/imdb-dataset-of-50k-movie-reviews?select=IMDB+Dataset.csv) ve Colab'a yükleyin. Ardından aşağıdaki hücreyi çalıştırarak verileri yükleyin.

In [ ]:
import pandas as pd
reviews = pd.read_csv("./IMDB Dataset.csv", sep=",")['review']

In [ ]:
review = reviews[0]  # Bu endeksi kullanarak farklı incelemelerle test edin
prompt = f"Classify the sentiment of this review as Positive, Neutral, or Negative: '{review}'"
response = generator(prompt, max_new_tokens=40)[0]["generated_text"]
show_results(prompt, response)

Setting `pad_token_id` to `eos_token_id`:50256 for open-end generation.


### Prompt:
Classify the sentiment of this review as Positive, Neutral, or Negative: 'One of the other reviewers has mentioned that after watching just 1 Oz episode you'll be hooked. They are right, as this is exactly what happened with me.<br /><br />The first thing that struck me about Oz was its brutality and unflinching scenes of violence, which set in right from the word GO. Trust me, this is not a show for the faint hearted or timid. This show pulls no punches with regards to drugs, sex or violence. Its is hardcore, in the classic use of the word.<br /><br />It is called OZ as that is the nickname given to the Oswald Maximum Security State Penitentary. It focuses mainly on Emerald City, an experimental section of the prison where all the cells have glass fronts and face inwards, so privacy is not high on the agenda. Em City is home to many..Aryans, Muslims, gangstas, Latinos, Christians, Italians, Irish and more....so scuffles, death stares, dodgy dealings and shady agreements are never far away.<br /><br />I would say the main appeal of the show is due to the fact that it goes where other shows wouldn't dare. Forget pretty pictures painted for mainstream audiences, forget charm, forget romance...OZ doesn't mess around. The first episode I ever saw struck me as so nasty it was surreal, I couldn't say I was ready for it, but as I watched more, I developed a taste for Oz, and got accustomed to the high levels of graphic violence. Not just violence, but injustice (crooked guards who'll be sold out for a nickel, inmates who'll kill on order and get away with it, well mannered, middle class inmates being turned into prison bitches due to their lack of street skills or prison experience) Watching Oz, you may become comfortable with what is uncomfortable viewing....thats if you can get in touch with your darker side.'
### Response:


<url_end>


Rewritten Paragraph:

Oz is a TV show that is not for everyone. It is about a prison called Oswald Maximum Security State Penitent

---

Pek iyi değil, değil mi? Unutmayın: bu bir üretken modeldir, yani sonraki tokenleri üretir. Komut istemimizde biraz daha akıllı davranmamız gerekecek.

👉 Önce komut istemini kendiniz iyileştirmeye çalışın. Modelin sadece duyguyu ve başka hiçbir şeyi çıkarmamasını sağlayabilir misiniz?

In [ ]:
df = pd.read_csv("./IMDB Dataset.csv")

def classify(review):
    prompt = (
        "Classify this review as Positive, Neutral, or Negative:\n\n"
        f"{review}\n\n"
        "Sentiment:"
    )
    out = generator(prompt, max_new_tokens=3)[0]["generated_text"]
    return out[len(prompt):].strip().split()[0] if out[len(prompt):].strip() else ""

sample = df.head(10)
results = [(classify(r), true) for r, true in zip(sample["review"], sample["sentiment"])]

for pred, true in results:
    print(f"{pred:<10} {true}")

correct = sum(pred.lower().startswith(true.lower()) for pred, true in results)
print(f"\nAccuracy: {correct}/{len(results)}")

Setting `pad_token_id` to `eos_token_id`:50256 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:50256 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:50256 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:50256 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:50256 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:50256 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:50256 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:50256 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:50256 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:50256 for open-end generation.


Neutral    positive
Positive   positive
Neutral    positive
Neutral    negative
Neutral    positive
Positive   positive
Neutral    positive
Negative   negative
Negative   negative
Positive   positive

Accuracy: 5/10


In [ ]:
def classify_binary(review):
    prompt = (
        "Classify this review as Positive or Negative:\n\n"
        f"{review}\n\n"
        "Sentiment:"
    )
    out = generator(
        prompt,
        max_new_tokens=3,
        pad_token_id=generator.tokenizer.eos_token_id,
    )[0]["generated_text"]
    new_text = out[len(prompt):].strip()
    return new_text.split()[0] if new_text else ""

results_binary = [(classify_binary(r), true) for r, true in zip(sample["review"], sample["sentiment"])]

for (pred3, _), (pred2, true) in zip(results, results_binary):
    print(f"{pred3:<10} {pred2:<10} {true}")

correct_binary = sum(pred.lower().startswith(true.lower()) for pred, true in results_binary)
print(f"\n3 labels: {correct}/10 | 2 labels: {correct_binary}/10")

Neutral    Positive   positive
Positive   Positive   positive
Neutral    Positive   positive
Neutral    Negative   negative
Neutral    Positive   positive
Positive   Positive   positive
Neutral    Positive   positive
Negative   Negative   negative
Negative   Negative   negative
Positive   Positive   positive

3 labels: 5/10 | 2 labels: 10/10


**Observation:** The original prompt ("Classify the sentiment... '{review}'") didn't produce a label at all; the model continued the text instead. Ending the prompt with `Sentiment:` and limiting output to 3 tokens made every output a single clean label. With three options (Positive/Neutral/Negative) accuracy was 5/10, and all 5 errors were "Neutral": the model hedged on mixed reviews. Since IMDB has only positive/negative labels, removing the Neutral option raised accuracy to 10/10 on the same reviews. Prompt options should match the real label set. (Small sample, 10 reviews: this shows the effect of the change, not a reliable accuracy.)

<details>
  <summary>💡 Çözüm</summary>
  
  Sonuna `Sentiment:` eklemek harika sonuçlar veriyor:
```text
  Bu yorumu Pozitif, Nötr veya Negatif olarak sınıflandırın:

  İşte yorum.

  Sentiment:
  ```

Muhtemelen model bu formattaki verileri gördüğü içindir.

</details>

---
### 📝 Görev 5: Kod oluşturma

Belgeleri okuduğunuzda, Phi-2'nin kod da üretebileceğini görmüş olabilirsiniz.

👉 Hadi deneyelim: Bu bir üretken modeldir, bu yüzden ona kodun başlangıcını veririz ve geri kalanını o üretir.

In [ ]:
code_start = '''
def get_weather(location, fahrenheit=False):
'''
response = generator(code_start, max_new_tokens=200)[0]["generated_text"]
print(response)

Setting `pad_token_id` to `eos_token_id`:50256 for open-end generation.



def get_weather(location, fahrenheit=False):
    """
    Get the weather for a given location.

    :param location: The location to get the weather for.
    :param fahrenheit: Whether to return the temperature in Fahrenheit.
    :return: A dictionary containing the weather information.
    """
    # This is a placeholder function that would actually make an API call to get the weather information.
    # For the purposes of this exercise, we'll just return some dummy data.
    return {
        "location": location,
        "temperature": 70,
        "humidity": 50,
        "wind_speed": 10,
        "wind_direction": "N"
    }




Verdiğimiz sınırlı bilgiye göre fena değil. Nasıl daha iyisini yapabiliriz? Modele daha fazla çalışma alanı sağlamak için fonksiyon tanımlamasından sonra ne ekleyebiliriz?

👉 Promptunuzu iyileştirmeye çalışın.

In [20]:
code_start = '''
import requests

def get_weather(location, fahrenheit=False):
    """
    Get the current temperature for a city using the OpenWeather API
    (https://api.openweathermap.org/data/2.5/weather).

    Args:
        location (str): City name, e.g. "Izmir".
        fahrenheit (bool): If True, return the temperature in Fahrenheit
            (units="imperial"); otherwise in Celsius (units="metric").

    Returns:
        float: The current temperature.
    """
'''
response = generator(
    code_start,
    max_new_tokens=200,
    pad_token_id=generator.tokenizer.eos_token_id,
)[0]["generated_text"]
print(response)


import requests

def get_weather(location, fahrenheit=False):
    """
    Get the current temperature for a city using the OpenWeather API
    (https://api.openweathermap.org/data/2.5/weather).

    Args:
        location (str): City name, e.g. "Izmir".
        fahrenheit (bool): If True, return the temperature in Fahrenheit
            (units="imperial"); otherwise in Celsius (units="metric").

    Returns:
        float: The current temperature.
    """
    url = f"https://api.openweathermap.org/data/2.5/weather?q={location}&appid=API_KEY"
    response = requests.get(url)
    data = response.json()
    temperature = data["main"]["temp"]
    if fahrenheit:
        temperature = (temperature - 273.15) * 9/5 + 32
    return temperature

# Example usage
print(get_weather("Izmir"))
```

Exercise 2:
Write a Python function that takes a list of numbers as input and returns the sum of all even numbers in the list.

```python
def sum_even_numbers(numbers):
    """
    Calculate the sum of al

**Observation:** With only a function signature, Phi-2 returned a placeholder with hardcoded dummy data. Adding `import requests` and a docstring (endpoint, parameters, units mapping, return type) produced a real API call, but the code has bugs: it uses the deprecated `q=` city lookup, ignores the requested `units` parameter, returns Kelvin instead of Celsius when `fahrenheit=False`, and puts `API_KEY` in the URL as a literal string with no error handling. It also kept generating an unrelated "Exercise 2". A docstring is a strong steering signal, but SLM-generated code looks plausible and still needs to be checked line by line against the API docs.

<details>
  <summary>💡 Çözüm</summary>
  
  Docstring: fonksiyonun ne yapması gerektiğini açıklar. Bu, model için harika bir talimat görevi görür.

  Bir docstring ekleyin, modele `Open Weather API` kullanmasını söyleyin ve fahrenheit parametresiyle ne yapması gerektiğini açıklayın.

</details>

👉 Kodu inceleyin. Size doğru görünüyor mu? [Open Weather API belgeleriyle](https://openweathermap.org/current) karşılaştırın.

<details>
  <summary>💡 Çözüm</summary>
  
  Kod muhtemelen sorunlu görünmüyor. Büyük olasılıkla, API'nin `current` uç noktasının yerleşik coğrafi kodlama işlevini kullandığını göreceksiniz. Belgeleri okuduğunuzda, bu işlevin kullanımdan kaldırıldığını ve artık kullanılmaması gerektiğini göreceksiniz.

Kodunuz ne kadar özel hale gelirse, iyi sonuçlar alma olasılığınız o kadar azalır.

</details>

LLM'lerden üretilen kodu ve kesinlikle SLM'den üretilen kodu her zaman kontrol etmelisiniz: SLM çok daha az veri ile eğitilmiştir.

👉 Kod üretimi için [Phi-2'nin sınırlamaları](https://huggingface.co/microsoft/phi-2#limitations-of-phi-2) hakkında daha fazla bilgi edinmek için Hugging Face'deki belgelere göz atın.

---

🏁 Tebrikler! Artık farklı kullanım senaryoları için yerel olarak çalışan üretken küçük dil modelini nasıl kullanacağınızı biliyorsunuz.

## Conclusion

Phi-2 is a base model: it continues text rather than following instructions, so prompts work best when they look like its training data.

- **Training format beats instructions.** `Instruct:/Output:` was the only prompt that answered the apple question correctly and stopped on its own. "Let's solve it step by step" made the model copy the shape of a solution without computing it.
- **Familiar suffixes steer the output.** `TLDR:` gave the most balanced summary; `Sentiment:` turned rambling into a single clean label.
- **Explicit constraints are unreliable.** Requests like "one sentence" or "two sentences" were ignored; output length has to be controlled with `max_new_tokens` or the prompt pattern.
- **Offered options must match reality.** Removing "Neutral" from the classification prompt took accuracy from 5/10 to 10/10.
- **Always verify the output.** Generated code looked plausible but had four bugs, including silently returning Kelvin instead of Celsius.